In [ ]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'        # Reduce TensorFlow log messages
os.environ["TF_DETERMINISTIC_OPS"] = "1"        # Enable deterministic operations for reproducibility
os.environ["TF_CUDNN_DETERMINISTIC"] = "1"      # Fix cudnn randomness for reproducibility
import random
import numpy as np
import tensorflow as tf
from pathlib import Path
from model import init_model, fun


In [ ]:
cell = "GM12878"
resolution = 10000    # Hi-C resolution (bp): 10000(10kb), 25000(25kb)
display_reso = int(resolution / 1000)
SEED = 42  # 5 sets of random seeds:42,43,44,45,46
# Update this root_dir to your own project path before running
root_dir = "/mnt/d/TAD-MultiOmicsNet"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


use_epi = True
use_dnabert = True
use_curv = True
use_transformer = False
use_maskconv = True
use_gaussian = False
use_adaptive_fuse = False
model_use = f"HiC_Ep_Dn_Cv_Mk_seed{SEED}"

In [ ]:
# ====================== Module 1: Robustness Prediction with Missing Features ======================
# Test model performance under different feature missing rates
# Add random mask to multi-omics features, generate prediction probabilities and save npy results

In [ ]:
chrs = ["20", "21", "22"]
missing_rates_all = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]

In [ ]:

def apply_bin_missing(feat, orig_mask, idx_missing):
    feat_corrupted = feat.copy()
    mask_corrupted = orig_mask.copy()
    if len(idx_missing) > 0:
        if mask_corrupted.ndim == 2:
            mask_corrupted[idx_missing, :] = 0.0
        else:
            mask_corrupted[idx_missing] = 0.0
    return feat_corrupted, mask_corrupted


print(f"\n===== {model_use} =====")
model = init_model(
        use_epi=use_epi,
        use_dnabert=use_dnabert,
        use_curv=use_curv,
        use_transformer=use_transformer,
        use_maskconv=use_maskconv,
        use_gaussian=use_gaussian,
        use_adaptive_fuse=use_adaptive_fuse
        )
weight_path = f'{root_dir}/{cell}/{display_reso}kb/weights/{model_use}_best.h5'
model.load_weights(weight_path)
print(f"✅ Load weights: {weight_path}")

epi_feat_all = np.load(f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/epi{cell}_predict_feat100.npy").astype("float32")
epi_mask_all = np.load(f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/epi{cell}_predict_mask100.npy").astype("float32")
dna_feat_all = np.load(f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/dna{cell}_predict_{display_reso}kb.npy").astype("float32")
dna_mask_all = np.load(f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/dna{cell}_predict_mask_{display_reso}kb.npy").astype("float32")
curv_feat_all = np.load(f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/curv{cell}_predict_feat100.npy").astype("float32")
curv_mask_all = np.load(f"{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/curv{cell}_predict_mask100.npy").astype("float32")

mr_list = missing_rates_all
for mr in mr_list:
    print(f"\n------ missing_rate = {mr:.2f} ------")
    out_dir = Path(
        f'{root_dir}/{cell}/{display_reso}kb/prediction_result/missingRobust/{model_use}_mr{mr:.2f}')
    out_dir.mkdir(parents=True, exist_ok=True)
    ptr = 0
    for chrom_idx, chrom in enumerate(chrs):
        npy_path = f'{root_dir}/{cell}/{display_reso}kb/samples-generate/predict/{cell}_{display_reso}kb_chr{chrom}-matrix.npy'
        pre_data = np.load(npy_path)
        sample_num = pre_data.shape[0]
        data_hic_raw = pre_data[:, :-1].reshape(-1, 10, 10, 1).astype("float32")
        data_hic = fun(data_hic_raw)

        epi_feat_chr = epi_feat_all[ptr:ptr + sample_num].copy()
        epi_mask_chr = epi_mask_all[ptr:ptr + sample_num].copy()
        dna_feat_chr = dna_feat_all[ptr:ptr + sample_num].copy()
        dna_mask_chr = dna_mask_all[ptr:ptr + sample_num].copy()
        curv_feat_chr = curv_feat_all[ptr:ptr + sample_num].copy()
        curv_mask_chr = curv_mask_all[ptr:ptr + sample_num].copy()

        # Each (mr,chrom) has an independent random seed to ensure reproducibility.
        sub_seed = SEED * 1000 + int(mr * 100) + chrom_idx
        rng = np.random.default_rng(seed=sub_seed)
        n_bin = epi_feat_chr.shape[0]
        n_missing = int(n_bin * mr)
        idx_missing = rng.choice(np.arange(n_bin), size=n_missing,
                                 replace=False) if n_missing > 0 else np.array([], dtype=int)

        epi_feat_cor, epi_mask_cor = apply_bin_missing(epi_feat_chr, epi_mask_chr, idx_missing)
        dna_feat_cor, dna_mask_cor = apply_bin_missing(dna_feat_chr, dna_mask_chr, idx_missing)
        curv_feat_cor, curv_mask_cor = apply_bin_missing(curv_feat_chr, curv_mask_chr, idx_missing)

        zero_epi = np.count_nonzero(epi_mask_cor == 0.0) / epi_mask_cor.shape[0]
        zero_dna = np.count_nonzero(dna_mask_cor == 0.0) / dna_mask_cor.shape[0]
        zero_curv = np.count_nonzero(curv_mask_cor == 0.0) / curv_mask_cor.shape[0]
        print(f"DEBUG mr={mr}, chr={chrom}, zero_mask_frac: epi={zero_epi:.3f}, dna={zero_dna:.3f}, curv={zero_curv:.3f}")

        inputs = [data_hic]
        inputs.extend([epi_feat_cor, epi_mask_cor])
        inputs.extend([dna_feat_cor, dna_mask_cor])
        inputs.extend([curv_feat_cor, curv_mask_cor])
        ptr += sample_num

        predictions = model.predict(inputs, batch_size=64, verbose=1)
        pred_prob = np.array(predictions.flatten())
        merged = np.concatenate([pre_data, pred_prob.reshape(-1, 1)], axis=1)
        save_path = str(out_dir / f"{display_reso}kb_{cell}_chr{chrom}-pred.npy")
        np.save(save_path, merged)
        print(f"{save_path}, shape={merged.shape}")

print("\nPrediction file for total missing rate has been generated!")


In [ ]:
# ====================== Module 2: Evaluation of Robustness Experiment ======================
# Load prediction npy files under different missing rates
# Calculate classification metrics: Precision, Recall, F1, AUPRC, MCC

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    average_precision_score, matthews_corrcoef,
)


mr_list = missing_rates_all
for mr in mr_list:
    print(f"\n{model_use}_mr{mr:.2f}")
    FIX_THRESH = 0.5
    file_list = [
        ("chr20", f"{root_dir}/{cell}/{display_reso}kb/prediction_result/missingRobust/{model_use}_mr{mr:.2f}/{display_reso}kb_{cell}_chr20-pred.npy"),
        ("chr21", f"{root_dir}/{cell}/{display_reso}kb/prediction_result/missingRobust/{model_use}_mr{mr:.2f}/{display_reso}kb_{cell}_chr21-pred.npy"),
        ("chr22", f"{root_dir}/{cell}/{display_reso}kb/prediction_result/missingRobust/{model_use}_mr{mr:.2f}/{display_reso}kb_{cell}_chr22-pred.npy"),
    ]

    chr_prf1 = []
    chr_auprc = []
    chr_mcc = []
    all_total_bins = 0

    print("==== Per-chromosome evaluation ====")
    for chr_name, path in file_list:
        arr = np.load(path)
        y_true = arr[:, -2]
        y_prob = arr[:, -1]
        y_pred = (y_prob >= FIX_THRESH)

        chr_total_bins = arr.shape[0]
        all_total_bins += chr_total_bins

        p = precision_score(y_true, y_pred, pos_label=1)
        r = recall_score(y_true, y_pred, pos_label=1)
        f1 = f1_score(y_true, y_pred, pos_label=1)
        mcc = matthews_corrcoef(y_true, y_pred)
        pos_num = np.sum(y_true == 1)
        pred_pos_num = np.sum(y_pred == 1)

        # AUPRC: skip if only one class exists
        if len(np.unique(y_true)) == 2:
            auprc = average_precision_score(y_true, y_prob)
        else:
            auprc = np.nan

        chr_prf1.append([p, r, f1])
        chr_auprc.append(auprc)
        chr_mcc.append(mcc)

        print(f"[{chr_name}] True boundaries:{pos_num};Pred boundaries:{pred_pos_num};P={p:.4f};R={r:.4f};F1={f1:.4f}")
        print(f"AUPRC={auprc:.4f};MCC={mcc:.4f}\n")

    # Chromosome average metrics
    chr_prf1_arr = np.array(chr_prf1)
    chr_avg_p = np.nanmean(chr_prf1_arr[:, 0])
    chr_avg_r = np.nanmean(chr_prf1_arr[:, 1])
    chr_avg_f1 = np.nanmean(chr_prf1_arr[:, 2])
    chr_avg_auprc = np.nanmean(chr_auprc)
    chr_avg_mcc = np.nanmean(chr_mcc)
    print("==== Chromosome average metrics (equal weight per chromosome) ====")
    print(f"Average P={chr_avg_p:.4f};Average R={chr_avg_r:.4f};Average F1={chr_avg_f1:.4f}")
    print(f"Average AUPRC={chr_avg_auprc:.4f};Average MCC={chr_avg_mcc:.4f}\n")